In [1]:
import pandas as pd
import networkx as nx
import pickle

# Load graph
with open('../data/user_graph.pkl', 'rb') as f:
    G = pickle.load(f)

# Load issue-level data
issue_summary = pd.read_csv('../data/issue_summary.csv')

# Load tweet -> issue mapping
tweet_issue_labels = pd.read_csv('../data/tweet_issue_labels.csv')

# Load conversation structure
conversation = pd.read_csv("../data/conversation_structure.csv")

print("Graph:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")
print("Issues:", issue_summary.shape)
print("Tweet-Issue Labels:", tweet_issue_labels.shape)
print("Conversation Structure:", conversation.shape)

Graph: 423345 nodes, 744835 edges
Issues: (22, 7)
Tweet-Issue Labels: (250000, 6)
Conversation Structure: (250000, 3)


In [2]:
print(type(list(G.nodes())[0]))

<class 'str'>


In [ ]:
print(tweet_issue_labels.columns)

Index(['id', 'user', 'hashtags', 'clean_text', 'issue_semantic', 'similarity'], dtype='str')


In [ ]:
import re

def clean_user(x):
    if isinstance(x, float):
        return str(int(x))
    
    if isinstance(x, str):
        x = x.strip()
        if x.startswith('{'):
            # Just extract id_str directly with regex
            m = re.search(r"'id_str'\s*:\s*'(\d+)'", x)
            if m:
                return m.group(1)
            # Fallback to numeric id field
            m = re.search(r"'id'\s*:\s*(\d+)", x)
            if m:
                return m.group(1)
        try:
            return str(int(float(x)))
        except (ValueError, TypeError):
            pass
    
    return str(x)

tweet_issue_labels['user'] = tweet_issue_labels['user'].apply(clean_user)

In [6]:
print(tweet_issue_labels.shape)
print(tweet_issue_labels.head())

(250000, 6)
                    id                 user  \
0  1801041792923578484   942869257108455424   
1  1801041792630227173  1461100431329796100   
2  1801041792592224521  1655734665955737600   
3  1801041791463866688  1771777682587713536   
4  1801041790952231228            874708668   

                                            hashtags  \
0                                                 []   
1                                                 []   
2  [{'indices': [171, 176], 'text': 'MAGA'}, {'in...   
3                                                 []   
4                                                 []   

                                          clean_text         issue_semantic  \
0  i cant imagine anyone actually feels this way....  Democracy in the U.S.   
1  voters can also sway me away from voting for s...             Gun policy   
2  can you name that amount of charges brought ag...  Democracy in the U.S.   
3  the fact remains that joe biden is simply a te...

In [7]:
# Check the actual types of the problematic rows
for val in tweet_issue_labels['user'].head(5):
    print(type(val), repr(val))

<class 'str'> '942869257108455424'
<class 'str'> '1461100431329796100'
<class 'str'> '1655734665955737600'
<class 'str'> '1771777682587713536'
<class 'str'> '874708668'


### Graph metrics at user level

In [8]:
sample_nodes = list(G.nodes())[:5]
for n in sample_nodes:
    print(type(n), repr(n))


<class 'str'> "{'id': 942869257108455424, 'id_str': '942869257108455424', 'url': 'https://twitter.com/B. G. Dalena', 'username': 'B. G. Dalena', 'rawDescription': 'Therapist. Philosophy, Epistemology, Politics, Neuroscience, Golf, Art, Sports, and a relentless pursuit of the truth.', 'created': datetime.datetime(2017, 12, 18, 21, 28, 43, tzinfo=datetime.timezone.utc), 'followersCount': 152, 'friendsCount': 246, 'statusesCount': 390, 'favouritesCount': 19, 'listedCount': 0, 'mediaCount': 1, 'location': 'San Diego, CA', 'profileImageUrl': 'https://pbs.twimg.com/profile_images/1613647617594130432/h3jM9GzR_normal.jpg', 'profileBannerUrl': 'PW', 'protected': 'PW', 'verified': False, 'blue': False, 'blueType': None, 'descriptionLinks': ['PW'], '_type': 'PW'}"
<class 'float'> 1.4835964632212644e+18
<class 'str'> "{'id': 1655734665955737600, 'id_str': '1655734665955737600', 'url': 'https://twitter.com/John', 'username': 'John', 'rawDescription': 'Raised in the beautiful wine country of Augusta

In [9]:
sample = list(G.nodes())[0]
print(repr(sample))
try:
    import ast
    parsed = ast.literal_eval(sample)
    print("Parsed OK:", parsed)
except Exception as e:
    print("Failed:", e)

"{'id': 942869257108455424, 'id_str': '942869257108455424', 'url': 'https://twitter.com/B. G. Dalena', 'username': 'B. G. Dalena', 'rawDescription': 'Therapist. Philosophy, Epistemology, Politics, Neuroscience, Golf, Art, Sports, and a relentless pursuit of the truth.', 'created': datetime.datetime(2017, 12, 18, 21, 28, 43, tzinfo=datetime.timezone.utc), 'followersCount': 152, 'friendsCount': 246, 'statusesCount': 390, 'favouritesCount': 19, 'listedCount': 0, 'mediaCount': 1, 'location': 'San Diego, CA', 'profileImageUrl': 'https://pbs.twimg.com/profile_images/1613647617594130432/h3jM9GzR_normal.jpg', 'profileBannerUrl': 'PW', 'protected': 'PW', 'verified': False, 'blue': False, 'blueType': None, 'descriptionLinks': ['PW'], '_type': 'PW'}"
Failed: malformed node or string on line 1: <ast.Call object at 0x3ab499750>


In [10]:
sample_nodes = list(G.nodes())[:10]
for n in sample_nodes:
    print(f"Input type: {type(n).__name__} | Output: {clean_user(n)}")

Input type: str | Output: 942869257108455424
Input type: float | Output: 1483596463221264384
Input type: str | Output: 1655734665955737600
Input type: float | Output: 1480159185606258688
Input type: str | Output: 1771777682587713536
Input type: float | Output: 254117355
Input type: str | Output: 874708668
Input type: float | Output: 3315264553
Input type: str | Output: 1667014094627520512
Input type: float | Output: 428454304


In [11]:
print(all(isinstance(clean_user(n), str) for n in list(G.nodes())[:2000]))
for n in list(G.nodes())[:10]:
    result = clean_user(n)
    print(f"Input type: {type(n).__name__} | Output: {result!r} | Output type: {type(result).__name__}")

True
Input type: str | Output: '942869257108455424' | Output type: str
Input type: float | Output: '1483596463221264384' | Output type: str
Input type: str | Output: '1655734665955737600' | Output type: str
Input type: float | Output: '1480159185606258688' | Output type: str
Input type: str | Output: '1771777682587713536' | Output type: str
Input type: float | Output: '254117355' | Output type: str
Input type: str | Output: '874708668' | Output type: str
Input type: float | Output: '3315264553' | Output type: str
Input type: str | Output: '1667014094627520512' | Output type: str
Input type: float | Output: '428454304' | Output type: str


In [12]:
# Compute graph metrics at user level

# Degree
degree_dict = dict(G.degree())

# PageRank
pagerank_dict = nx.pagerank(G)

# Betweenness
betweenness_dict = nx.betweenness_centrality(G, k=1000)  # sample for speed

degree_dict = {clean_user(k): v for k, v in dict(G.degree()).items()}
pagerank_dict_clean = {clean_user(k): v for k, v in nx.pagerank(G).items()}
betweenness_dict_clean = {clean_user(k): v for k, v in betweenness_dict.items()}

# print dictionary
print("Degree dict sample:", list(degree_dict.items())[:5])
print("PageRank dict sample:", list(pagerank_dict_clean.items())[:5])
print("Betweenness dict sample:", list(betweenness_dict_clean.items())[:5])


Degree dict sample: [('942869257108455424', 3), ('1483596463221264384', 377), ('1655734665955737600', 2), ('1480159185606258688', 3), ('1771777682587713536', 3)]
PageRank dict sample: [('942869257108455424', 7.479406622749844e-07), ('1483596463221264384', 0.00016108618175450173), ('1655734665955737600', 1.991766942802008e-06), ('1480159185606258688', 1.4994594037872515e-06), ('1771777682587713536', 1.7325421129763558e-06)]
Betweenness dict sample: [('942869257108455424', 2.211641201574713e-10), ('1483596463221264384', 0.0005150534686952047), ('1655734665955737600', 1.5046900503846763e-06), ('1480159185606258688', 3.66844143111227e-08), ('1771777682587713536', 1.0038373443151555e-06)]


In [13]:
# Convert to dataframe
user_features = pd.DataFrame({
    'user': list(degree_dict.keys()),
    'degree': list(degree_dict.values()),
    'pagerank': [pagerank_dict_clean[u] for u in degree_dict.keys()],
    'betweenness': [betweenness_dict_clean[u] for u in degree_dict.keys()]
})

user_features.head()

,user,degree,pagerank,betweenness
0,942869257108455424,3,7.479407e-07,2.211641e-10
1,1483596463221264384,377,1.610862e-04,5.150535e-04
2,1655734665955737600,2,1.991767e-06,1.504690e-06
3,1480159185606258688,3,1.499459e-06,3.668441e-08
4,1771777682587713536,3,1.732542e-06,1.003837e-06


In [14]:
# user_features['user'] = user_features['user'].apply(clean_user)
print(user_features.shape)
print(user_features.head())

(342303, 4)
                  user  degree      pagerank   betweenness
0   942869257108455424       3  7.479407e-07  2.211641e-10
1  1483596463221264384     377  1.610862e-04  5.150535e-04
2  1655734665955737600       2  1.991767e-06  1.504690e-06
3  1480159185606258688       3  1.499459e-06  3.668441e-08
4  1771777682587713536       3  1.732542e-06  1.003837e-06


### Avg degree of users per issue

In [15]:
# Merge user features with tweet -> issue mapping
df = tweet_issue_labels.merge(user_features, on='user', how='left')

# Avg degree per issue
avg_degree_per_issue = df.groupby('issue_semantic')['degree'].mean().reset_index()
avg_degree_per_issue.rename(columns={'degree': 'avg_user_degree'}, inplace=True)

### Hashtag node degree

In [17]:
# Extract hashtag nodes
hashtag_nodes = [n for n in G.nodes() if str(n).startswith("hashtag_")]

# Get their degree
hashtag_degree = {n: G.degree(n) for n in hashtag_nodes}

# Map hashtags → issues via tweet_issue_labels
# (assuming tweet_issue_labels has hashtags column or similar mapping)

# explode hashtags per tweet
# if 'hashtags' in tweet_issue_labels.columns:
#     print('hashtags' in tweet_issue_labels.columns)
#     print(tweet_issue_labels['hashtags'].head())
#     tweet_issue_labels['hashtags'] = tweet_issue_labels['hashtags'].fillna('').apply(lambda x: x.split())
#     temp = tweet_issue_labels.explode('hashtags')
#     temp['hashtag_node'] = temp['hashtags'].str.lower().apply(lambda x: f"hashtag_{x}")
#     temp['hashtag_degree'] = temp['hashtag_node'].map(hashtag_degree)
#     hashtag_degree_per_issue = temp.groupby('issue')['hashtag_degree'].mean().reset_index()  # ✅ indented
#     hashtag_degree_per_issue.rename(columns={'hashtag_degree': 'avg_hashtag_degree'}, inplace=True)  # ✅ indented
#     print(temp['hashtag_node'].head(10))
#     print(temp['hashtag_degree'].head(10))
# else:
#     print("No 'hashtags' column found in tweet_issue_labels")
#     hashtag_degree_per_issue = pd.DataFrame(columns=['issue', 'avg_hashtag_degree'])

# hashtag_degree_per_issue = temp.groupby('issue')['hashtag_degree'].mean().reset_index()
# hashtag_degree_per_issue.rename(columns={'hashtag_degree': 'avg_hashtag_degree'}, inplace=True)

In [18]:
print(f"Total hashtag nodes: {len(hashtag_nodes)}")
print(list(hashtag_nodes)[:5])

Total hashtag nodes: 10844
['hashtag_maga', 'hashtag_trump2024', 'hashtag_internacional', 'hashtag_trumpbordercrisis', 'hashtag_biden']


In [19]:
print('hashtags' in tweet_issue_labels.columns)

True


In [20]:
# extract user-hashtag edges
user_hashtag = []

for u, v, data in G.edges(data=True):
    if data.get('type') == 'hashtag':
        if str(v).startswith("hashtag_"):
            user_hashtag.append((str(u), str(v)))
        elif str(u).startswith("hashtag_"):
            user_hashtag.append((str(v), str(u)))

user_hashtag_df = pd.DataFrame(user_hashtag, columns=['user', 'hashtag_node'])

# clean hashtag
user_hashtag_df['user'] = user_hashtag_df['user'].apply(clean_user)  # add this line

In [21]:
# merge with tweet - issue
# ensure user type match
user_hashtag_df['user'] = user_hashtag_df['user'].astype(str)
tweet_issue_labels['user'] = tweet_issue_labels['user'].astype(str)

user_issue_hashtag = user_hashtag_df.merge(
    tweet_issue_labels[['user', 'issue_semantic']],
    on='user',
    how='left'
)

In [22]:
# get hashtag degree
hashtag_degree = {n: G.degree(n) for n in G.nodes() if str(n).startswith("hashtag_")}

user_issue_hashtag['hashtag_degree'] = user_issue_hashtag['hashtag_node'].map(hashtag_degree)

In [23]:
# aggregate per issue
hashtag_degree_per_issue = (
    user_issue_hashtag
    .groupby('issue_semantic')['hashtag_degree']
    .mean()
    .reset_index()
    .rename(columns={'hashtag_degree': 'avg_hashtag_degree'})
)

print(hashtag_degree_per_issue.head())

                                  issue_semantic  avg_hashtag_degree
0                                       Abortion          273.514127
1                                 Climate change          254.070762
2                                          Crime          213.911880
3                          Democracy in the U.S.          383.789908
4  Distribution of income and wealth in the U.S.          196.459947


In [24]:
graph_users = set(user_hashtag_df['user'])
label_users = set(tweet_issue_labels['user'])
print(f"Users in graph hashtag edges: {len(graph_users)}")
print(f"Users in tweet_issue_labels: {len(label_users)}")
print(f"Overlap: {len(graph_users & label_users)}")

Users in graph hashtag edges: 11195
Users in tweet_issue_labels: 129814
Overlap: 11195


In [25]:
edge_types = set(data.get('type') for u, v, data in G.edges(data=True))
print(edge_types)

{'reply', 'hashtag', 'mention', 'conversation'}


In [26]:
for u, v, data in list(G.edges(data=True))[:5]:
    print(f"u: {repr(u)[:50]} | v: {repr(v)[:50]} | data: {data}")

u: "{'id': 942869257108455424, 'id_str': '94286925710 | v: 1.4835964632212644e+18 | data: {'type': 'reply'}
u: "{'id': 942869257108455424, 'id_str': '94286925710 | v: 'lukepbeasley' | data: {'type': 'mention'}
u: "{'id': 942869257108455424, 'id_str': '94286925710 | v: 'conv_1.8006810262489418e+18' | data: {'type': 'conversation'}
u: 1.4835964632212644e+18 | v: "{'id': 1586776616247742466, 'id_str': '1586776616 | data: {'type': 'reply'}
u: 1.4835964632212644e+18 | v: "{'id': 1683804812872151040, 'id_str': '1683804812 | data: {'type': 'reply'}


In [27]:
# Check raw edge data before any cleaning
for u, v, data in G.edges(data=True):
    if data.get('type') == 'hashtag':
        print(f"u: {repr(u)[:80]}")
        print(f"v: {repr(v)[:80]}")
        print(f"data: {data}")
        break

u: "{'id': 1655734665955737600, 'id_str': '1655734665955737600', 'url': 'https://tw
v: 'hashtag_maga'
data: {'type': 'hashtag'}


In [28]:
sample_user_raw = "{'id': 1655734665955737600, 'id_str': '1655734665955737600'..."
cleaned = clean_user(list(G.edges(data=True))[0][0])  # get first hashtag edge user
print(f"Cleaned user: {cleaned}")
print(f"In tweet_issue_labels: {cleaned in set(tweet_issue_labels['user'])}")

Cleaned user: 942869257108455424
In tweet_issue_labels: True


In [29]:
print("user_hashtag_df shape:", user_hashtag_df.shape)
print("Sample users from user_hashtag_df:")
print(user_hashtag_df['user'].head())

print("\nSample users from tweet_issue_labels:")
print(tweet_issue_labels['user'].head())

print("\nuser_issue_hashtag shape after merge:", user_issue_hashtag.shape)

user_hashtag_df shape: (36475, 2)
Sample users from user_hashtag_df:
0    1655734665955737600
1    1655734665955737600
2     796186301502488578
3               52600406
4     755408357134131200
Name: user, dtype: str

Sample users from tweet_issue_labels:
0     942869257108455424
1    1461100431329796100
2    1655734665955737600
3    1771777682587713536
4              874708668
Name: user, dtype: str

user_issue_hashtag shape after merge: (213631, 4)


### Centrality (issue-level)

In [31]:
# Avg PageRank per issue
avg_pagerank_per_issue = df.groupby('issue_semantic')['pagerank'].mean().reset_index()
avg_pagerank_per_issue.rename(columns={'pagerank': 'avg_pagerank'}, inplace=True)

# Avg Betweenness per issue
avg_betweenness_per_issue = df.groupby('issue_semantic')['betweenness'].mean().reset_index()
avg_betweenness_per_issue.rename(columns={'betweenness': 'avg_betweenness'}, inplace=True)

### Diffusion

#### Reply chains (distinct conversations)

In [34]:
print(conversation.columns.tolist())
print(conversation.head(2))

['id', 'conversationId', 'in_reply_to_status_id_str']
                    id  conversationId  in_reply_to_status_id_str
0  1801041792923578484    1.800681e+18               1.800681e+18
1  1801041792630227173    1.801042e+18                        NaN


In [35]:
# merge conversation info
conv_df = tweet_issue_labels.merge(conversation, on='id', how='left')

reply_chains_per_issue = conv_df.groupby('issue_semantic')['conversationId'].nunique().reset_index()
reply_chains_per_issue.rename(columns={'conversationId': 'num_conversations'}, inplace=True)

#### Conversation depth (max chain length)

In [36]:
# Build depth per conversation

# parent mapping
parent_map = dict(zip(conversation['id'], conversation['in_reply_to_status_id_str']))

def get_depth(tweet_id, parent_map, memo, visited=set()):
    if tweet_id in memo:
        return memo[tweet_id]
    if tweet_id in visited:
        return 1  # break loop

    visited.add(tweet_id)
    parent = parent_map.get(tweet_id)
    
    if pd.isna(parent) or parent not in parent_map:
        memo[tweet_id] = 1
    else:
        memo[tweet_id] = 1 + get_depth(parent, parent_map, memo)
    
    return memo[tweet_id]

memo = {}
conversation['depth'] = conversation['id'].apply(lambda x: get_depth(x, parent_map, memo))

# max depth per conversation
conv_depth = conversation.groupby('conversationId')['depth'].max().reset_index()
conv_depth.rename(columns={'depth': 'conv_depth'}, inplace=True)

#### Avg depth per issue

In [38]:
conv_issue = conv_df[['issue_semantic', 'conversationId']].drop_duplicates()
conv_issue = conv_issue.merge(conv_depth, on='conversationId', how='left')

avg_depth_per_issue = conv_issue.groupby('issue_semantic')['conv_depth'].mean().reset_index()

#### Spread

In [41]:
issue_summary['spread'] = issue_summary['Users'] / issue_summary['Tweets']
spread_per_issue = issue_summary[['Issue', 'spread']]

### User influence

In [42]:
# Avg PageRank (computed above)
avg_influence_per_issue = avg_pagerank_per_issue.copy()
avg_influence_per_issue.rename(columns={'avg_pagerank': 'avg_user_influence'}, inplace=True)

### Save Features

In [44]:
# standardize issue column name across all dataframes
issue_summary.rename(columns={'Issue': 'issue'}, inplace=True)
avg_degree_per_issue.rename(columns={'issue_semantic': 'issue'}, inplace=True)
hashtag_degree_per_issue.rename(columns={'issue_semantic': 'issue'}, inplace=True)
avg_pagerank_per_issue.rename(columns={'issue_semantic': 'issue'}, inplace=True)
avg_betweenness_per_issue.rename(columns={'issue_semantic': 'issue'}, inplace=True)
reply_chains_per_issue.rename(columns={'issue_semantic': 'issue'}, inplace=True)
avg_depth_per_issue.rename(columns={'issue_semantic': 'issue'}, inplace=True)
avg_influence_per_issue.rename(columns={'issue_semantic': 'issue'}, inplace=True)

In [45]:
features = issue_summary[['issue', 'Tweets', 'Users']]

features = features.merge(avg_degree_per_issue, on='issue', how='left')
features = features.merge(hashtag_degree_per_issue, on='issue', how='left')
features = features.merge(avg_pagerank_per_issue, on='issue', how='left')
features = features.merge(avg_betweenness_per_issue, on='issue', how='left')
features = features.merge(reply_chains_per_issue, on='issue', how='left')
features = features.merge(avg_depth_per_issue, on='issue', how='left')

features['spread'] = features['Users'] / features['Tweets']

In [46]:
pd.set_option('display.max_columns', None)
print(features)

print(features.head())
print("\nColumns:\n", features.columns)
print("\nShape:", features.shape)

                                                issue  Tweets  Users  \
0                               Democracy in the U.S.   91164  82224   
1                                               Crime   25422  23300   
2                                         Immigration   18946  16550   
3   Types of Supreme Court justices candidates wou...   17227  16419   
4                          The federal budget deficit   13864  13342   
5   Situation in Middle East between Israelis and ...   11432   9634   
6                                          Gun policy    9911   9304   
7                                Relations with China    8768   8459   
8                                      Climate change    7585   7269   
9                                  Transgender rights    6497   6233   
10                                        The economy    5086   4868   
11                                           Abortion    4952   4786   
12                    Terrorism and national security    4780   

In [49]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        'Avg user degree', 'Avg hashtag degree',
        'Avg pagerank', 'Avg betweenness',
        'Reply chains', 'Spread (users/tweets)'
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

cols_config = [
    ('avg_user_degree',    1, 1, '#378ADD'),
    ('avg_hashtag_degree', 1, 2, '#1D9E75'),
    ('avg_pagerank',       2, 1, '#7F77DD'),
    ('avg_betweenness',    2, 2, '#D85A30'),
    ('num_conversations',  3, 1, '#EF9F27'),
    ('spread',             3, 2, '#5DCAA5'),
]

for col, row, col_n, color in cols_config:
    fig.add_trace(
        go.Bar(
            x=features['issue'],
            y=features[col],
            marker_color=color,
            name=col,
            showlegend=False
        ),
        row=row, col=col_n
    )

fig.update_layout(
    height=900,
    title_text='Graph features by issue',
    title_font_size=16,
    template='plotly_white',
)
fig.update_xaxes(tickangle=45, tickfont_size=9)
fig.show()

In [50]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        'Avg user degree', 'Avg hashtag degree',
        'Avg pagerank', 'Avg betweenness',
        'Reply chains', 'Spread (users/tweets)'
    ],
    vertical_spacing=0.08,
    horizontal_spacing=0.25
)

cols_config = [
    ('avg_user_degree',    1, 1, '#378ADD'),
    ('avg_hashtag_degree', 1, 2, '#1D9E75'),
    ('avg_pagerank',       2, 1, '#7F77DD'),
    ('avg_betweenness',    2, 2, '#D85A30'),
    ('num_conversations',  3, 1, '#EF9F27'),
    ('spread',             3, 2, '#5DCAA5'),
]

for col, row, col_n, color in cols_config:
    sorted_df = features.sort_values(col, ascending=True)
    fig.add_trace(
        go.Bar(
            y=sorted_df['issue'],
            x=sorted_df[col],
            orientation='h',
            marker_color=color,
            showlegend=False
        ),
        row=row, col=col_n
    )

fig.update_layout(
    height=1800,
    title_text='Graph features by issue',
    title_font_size=16,
    template='plotly_white',
)
fig.update_yaxes(tickfont_size=10)
fig.update_xaxes(tickfont_size=10)
fig.show()

In [52]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

features['avg_betweenness_scaled'] = features['avg_betweenness'] * 1e6
features['avg_pagerank_scaled'] = features['avg_pagerank'] * 1e6

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Avg hashtag degree', 'Avg betweenness (×10⁻⁶)',
        'Avg pagerank (×10⁻⁶)', 'Reply chains'
    ],
    vertical_spacing=0.08,
    horizontal_spacing=0.25
)

cols_config = [
    ('avg_hashtag_degree',    1, 1, '#1D9E75'),
    ('avg_betweenness_scaled', 1, 2, '#D85A30'),
    ('avg_pagerank_scaled',    2, 1, '#7F77DD'),
    ('num_conversations',      2, 2, '#EF9F27'),
]

for col, row, col_n, color in cols_config:
    sorted_df = features.sort_values(col, ascending=True)
    fig.add_trace(
        go.Bar(
            y=sorted_df['issue'],
            x=sorted_df[col],
            orientation='h',
            marker_color=color,
            showlegend=False
        ),
        row=row, col=col_n
    )

fig.update_layout(
    height=1200,
    title_text='Graph features by issue',
    title_font_size=16,
    template='plotly_white',
)
fig.update_yaxes(tickfont_size=10)
fig.update_xaxes(tickfont_size=10)
fig.show()